# 08 SGX Paper Tables

# ShiftGuard-X — experimental research notebook

**Research status:** Proof of concept only. No paper performance results are claimed. Data are synthetic by default; independent real-data evaluation is required.

**Colab:** Runtime > Run all. No paid API or GPU required. Each notebook is self-contained and recreates `sgx_core.py`.

In [ ]:
# Standalone notebook: writes the same versioned research core locally.
from pathlib import Path
CORE_SOURCE = '"""ShiftGuard-X experimental byte-exact compression research core, v0.1.\nThis prototype is NOT an established novel codec or a benchmarked improvement.\nDependencies: Python standard library; optional zstandard, lz4.frame, brotli.\n"""\nimport bz2, hashlib, json, lzma, math, os, struct, time, zlib\nfrom collections import Counter\nfrom pathlib import Path\nMAGIC=b\'SGX1\'\nTRANSFORMS=(\'identity\',\'delta1\',\'delta2\',\'xor1\')\nBACKENDS=(\'raw\',\'zlib\',\'bz2\',\'lzma\',\'zstd\',\'lz4\',\'brotli\')\nHEADER=struct.Struct(\'<4sBBIII\')  # magic, transform, backend, uncompressed bytes, payload bytes, CRC32\n\ndef transform(data, mode):\n    if mode==\'identity\': return data\n    out=bytearray(len(data)); a=b=0\n    for i,x in enumerate(data):\n        if mode==\'delta1\': y=(x-a)&255\n        elif mode==\'delta2\': y=(x-(2*a-b))&255\n        elif mode==\'xor1\': y=x^a\n        else: raise ValueError(mode)\n        out[i]=y; b,a=a,x\n    return bytes(out)\n\ndef inverse(residual,mode):\n    if mode==\'identity\':return residual\n    out=bytearray(len(residual)); a=b=0\n    for i,y in enumerate(residual):\n        if mode==\'delta1\': x=(y+a)&255\n        elif mode==\'delta2\':x=(y+2*a-b)&255\n        elif mode==\'xor1\':x=y^a\n        else:raise ValueError(mode)\n        out[i]=x;b,a=a,x\n    return bytes(out)\n\ndef supported_backends():\n    names=[\'raw\',\'zlib\',\'bz2\',\'lzma\']\n    for n,mod in [(\'zstd\',\'zstandard\'),(\'lz4\',\'lz4.frame\'),(\'brotli\',\'brotli\')]:\n        try: __import__(mod.split(\'.\')[0]);names.append(n)\n        except ImportError: pass\n    return tuple(names)\n\ndef compress(data,backend):\n    if backend==\'raw\':return data\n    if backend==\'zlib\':return zlib.compress(data,1)\n    if backend==\'bz2\':return bz2.compress(data,compresslevel=1)\n    if backend==\'lzma\':return lzma.compress(data,preset=0)\n    if backend==\'zstd\':\n        import zstandard;return zstandard.ZstdCompressor(level=1).compress(data)\n    if backend==\'lz4\':\n        import lz4.frame;return lz4.frame.compress(data,compression_level=0)\n    if backend==\'brotli\':\n        import brotli;return brotli.compress(data,quality=1)\n    raise ValueError(backend)\n\ndef decompress(data,backend):\n    if backend==\'raw\':return data\n    if backend==\'zlib\':return zlib.decompress(data)\n    if backend==\'bz2\':return bz2.decompress(data)\n    if backend==\'lzma\':return lzma.decompress(data)\n    if backend==\'zstd\':\n        import zstandard;return zstandard.ZstdDecompressor().decompress(data)\n    if backend==\'lz4\':\n        import lz4.frame;return lz4.frame.decompress(data)\n    if backend==\'brotli\':\n        import brotli;return brotli.decompress(data)\n    raise ValueError(backend)\n\ndef shannon_bpb(data):\n    if not data:return 0.\n    counts=Counter(data);n=len(data)\n    return -sum(v/n*math.log2(v/n) for v in counts.values())\n\ndef js_divergence(a,b):\n    if not a or not b:return 0.\n    pa=Counter(a);pb=Counter(b);na=len(a);nb=len(b)\n    v=0.\n    for k in pa.keys()|pb.keys():\n        p=pa[k]/na;q=pb[k]/nb;m=(p+q)/2\n        if p:v+=0.5*p*math.log2(p/m)\n        if q:v+=0.5*q*math.log2(q/m)\n    return v\n\ndef encode_block(data,mode=\'identity\',backend=\'zlib\'):\n    if len(data)>2**32-1:raise ValueError(\'block too large\')\n    t=transform(data,mode);payload=compress(t,backend)\n    modes=list(TRANSFORMS)\n    if backend not in supported_backends():raise ValueError(\'backend unavailable\')\n    return HEADER.pack(MAGIC,modes.index(mode),BACKENDS.index(backend),len(data),len(payload),zlib.crc32(data))+payload\n\ndef decode_block(blob):\n    if len(blob)<HEADER.size:raise ValueError(\'truncated header\')\n    magic,mi,bi,n,ln,crc=HEADER.unpack_from(blob)\n    if magic!=MAGIC:raise ValueError(\'invalid magic\')\n    if mi>=len(TRANSFORMS) or bi>=len(BACKENDS):raise ValueError(\'unknown transform/backend\')\n    if len(blob)!=HEADER.size+ln:raise ValueError(\'invalid payload length\')\n    residual=decompress(blob[HEADER.size:],BACKENDS[bi]);data=inverse(residual,TRANSFORMS[mi])\n    if len(data)!=n or zlib.crc32(data)!=crc:raise ValueError(\'length/CRC mismatch\')\n    return data\n\ndef chunks(data,n=65536):\n    if n<1:raise ValueError(\'chunk size must be positive\')\n    return [data[i:i+n] for i in range(0,len(data),n)] or [b\'\']\n\ndef policy_encode(block,policy=\'fixed\',backends=(\'raw\',\'zlib\'),lambda_mbps=0.0,probe_bytes=4096,previous=None,drift=0.,threshold=0.08):\n    """Return (encoded block, info). All times include decision overhead.\n    \'cost\': sample-transform screening then one full encode, conservative RAW fallback.\n    \'drift\': reuse previous action if not shifted, otherwise run cost policy.\n    \'oracle\': exhaustive action search; evaluation-only and incurs real search time.\n    """\n    start=time.perf_counter()\n    valid=[b for b in backends if b in supported_backends()]\n    if not valid: raise ValueError(\'no backends\')\n    actions=[(m,b) for m in TRANSFORMS for b in valid]\n    evaluated=0\n    if policy==\'fixed\': chosen=(\'identity\',\'zlib\' if \'zlib\' in valid else valid[0]);encoded=encode_block(block,*chosen); evaluated=1\n    elif policy==\'oracle\':\n        candidates=[]\n        for m,b in actions:\n            tic=time.perf_counter();out=encode_block(block,m,b);sec=time.perf_counter()-tic\n            candidates.append((len(out)+lambda_mbps*sec,m,b,out))\n            evaluated+=1\n        _,m,b,encoded=min(candidates,key=lambda z:z[0]);chosen=(m,b)\n    elif policy in (\'cost\',\'drift\',\'entropy\'):\n        if policy==\'drift\' and previous is not None and drift<threshold and previous in actions:\n            chosen=previous\n        else:\n            sample=block[:min(len(block),probe_bytes)]\n            scores=[]\n            for m,b in actions:\n                tic=time.perf_counter()\n                residual=transform(sample,m)\n                if policy==\'entropy\':score=shannon_bpb(residual)*len(sample)/8\n                else:score=len(compress(residual,b))\n                sec=time.perf_counter()-tic\n                scores.append((score+lambda_mbps*sec,m,b));evaluated+=1\n            _,m,b=min(scores,key=lambda z:z[0]);chosen=(m,b)\n        encoded=encode_block(block,*chosen)\n        if \'raw\' in valid and len(encoded)>HEADER.size+len(block):\n            encoded=encode_block(block,\'identity\',\'raw\');chosen=(\'identity\',\'raw\')\n    else:raise ValueError(policy)\n    seconds=time.perf_counter()-start\n    return encoded, {\'mode\':chosen[0],\'backend\':chosen[1],\'original_bytes\':len(block),\'encoded_bytes\':len(encoded),\'elapsed_s\':seconds,\'probes\':evaluated,\'drift\':drift}\n\ndef run_stream(blocks,policy=\'fixed\',backends=(\'raw\',\'zlib\'),threshold=0.08):\n    records=[];encoded=[];prior=None;previous_block=None\n    for i,b in enumerate(blocks):\n        drift=js_divergence(previous_block[:4096],b[:4096]) if previous_block is not None else 1.\n        blob,meta=policy_encode(b,policy,backends=backends,previous=prior,drift=drift,threshold=threshold)\n        assert decode_block(blob)==b\n        meta[\'block\']=i;records.append(meta);encoded.append(blob)\n        prior=(meta[\'mode\'],meta[\'backend\']);previous_block=b\n    return records,encoded\n\ndef synthetic_data(seed=2026):\n    import random\n    rng=random.Random(seed)\n    text=(b\'INFO 200 GET /api/items user_id=1234 status=ok\\n\')*2500\n    random_bytes=bytes(rng.randrange(256) for _ in range(100000))\n    walk=bytearray();v=127\n    for _ in range(100000):v=(v+rng.choice([-1,0,0,0,1]))&255;walk.append(v)\n    structured=b\'\'.join((f\'{{"id":{i},"status":"ok","region":"apac"}}\\n\').encode() for i in range(4000))\n    return {\'log\':text,\'random\':random_bytes,\'sensor_walk\':bytes(walk),\'jsonl\':structured}\n\ndef benchmark(data,policies=(\'fixed\',\'entropy\',\'cost\',\'drift\'),block_size=32768,backends=(\'raw\',\'zlib\')):\n    results=[];bb=chunks(data,block_size)\n    for p in policies:\n        records,blobs=run_stream(bb,p,backends)\n        t=sum(z[\'elapsed_s\'] for z in records);size=sum(map(len,blobs))\n        results.append({\'policy\':p,\'raw_bytes\':len(data),\'compressed_bytes\':size,\'ratio\':len(data)/size if size else 0,\'bits_per_byte\':8*size/len(data) if data else 0,\'encode_s\':t,\'encode_mib_s\':len(data)/1048576/t if t else 0,\'probes\':sum(z[\'probes\'] for z in records)})\n    return results\n\ndef save_json(path,obj):\n    Path(path).parent.mkdir(parents=True,exist_ok=True)\n    Path(path).write_text(json.dumps(obj,indent=2,default=str))\n'
Path('sgx_core.py').write_text(CORE_SOURCE, encoding='utf-8')
import importlib, sgx_core as sgx
importlib.reload(sgx)
from pathlib import Path
OUT=Path('sgx_outputs'); OUT.mkdir(exist_ok=True)
print('Available backends:',sgx.supported_backends())


## Goal: consolidate completed outputs without inventing missing results or treating synthetic pilots as publication evidence.

In [ ]:
import csv, json
names=['00_correctness.json','01_data_audit.csv','02_fixed_baselines.csv','03_residual_coding.csv','04_selector.csv','05_drift.csv','06_ablations.csv','07_generalization.csv']
manifest=[]
for name in names:
 p=OUT/name
 manifest.append({'file':name,'present':p.exists(),'bytes':p.stat().st_size if p.exists() else None})
print(*manifest,sep='\n')
sgx.save_json(OUT/'08_manifest.json',{'artifacts':manifest,'research_status':'Pilot/synthetic only; no validated novelty or journal-performance claims','limitations':['No independent external datasets by default','No proven regret bound','No matched-hardware multirun statistics']})
print('Next: upload independent corpora, freeze benchmarks, run repetitions, compare state-of-art published codec implementations.')